# 1.9 Algorithmic stability

If removing one training point barely changes the model, the model generalizes: stability implies generalization.

Stability bounds the learning algorithm rather than the entire hypothesis class. The computation below uses leave-one-out sensitivity as an empirical feel for the stability constant and contrasts stable estimators with 1-NN-style instability.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(16)


def theory_ladder(topic):
    """Return compact D1--D5 inputs for F16 theory notebooks."""
    if topic == "srm":
        return [
            {"name": "D1 lesson four-rung ladder", "emp": np.array([0.30, 0.15, 0.07, 0.05]), "H": np.array([2, 16, 256, 65536]), "m": 200, "delta": 0.05},
            {"name": "D2 six nested polynomial rungs", "emp": np.array([0.34, 0.21, 0.13, 0.09, 0.075, 0.07]), "H": np.array([2, 8, 32, 128, 512, 2048]), "m": 200, "delta": 0.05},
            {"name": "D3 larger sample shifts upward", "emp": np.array([0.30, 0.15, 0.07, 0.04, 0.035]), "H": np.array([2, 16, 256, 65536, 1048576]), "m": 2000, "delta": 0.05},
            {"name": "D4 noisy empirical errors", "emp": np.array([0.31, 0.19, 0.12, 0.115, 0.13]), "H": np.array([2, 16, 256, 4096, 65536]), "m": 300, "delta": 0.05},
            {"name": "D5 bad ordering stress case", "emp": np.array([0.28, 0.10, 0.03, 0.02, 0.00]), "H": np.array([4, 4096, 64, 1048576, 256]), "m": 120, "delta": 0.05},
        ]
    if topic == "regularization":
        x = np.linspace(-1.0, 1.0, 25)
        base = 1.0 + 2.0 * x - 1.5 * x ** 2
        return [
            {"name": "D1 lesson 3-row regression", "X": np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]), "y": np.array([1.0, 2.0, 2.0]), "lambdas": np.array([0.0, 1.0, 10.0, 100.0])},
            {"name": "D2 dense lambda path", "X": np.column_stack([np.ones_like(x), x]), "y": 1.0 + 2.0 * x + 0.15 * np.sin(7.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D3 higher polynomial capacity", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3, x ** 4, x ** 5]), "y": base + 0.10 * np.sin(9.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D4 noisy labels", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3]), "y": base + rng.normal(0.0, 0.18, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D5 unscaled-feature trap", "X": np.column_stack([np.ones_like(x), x, 100.0 * x ** 2]), "y": base + rng.normal(0.0, 0.10, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
        ]
    if topic == "stability":
        sample = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
        long_sample = np.linspace(0.0, 1.0, 50)
        return [
            {"name": "D1 lesson 5-number mean", "sample": sample, "kind": "mean"},
            {"name": "D2 m=50 bounded-range mean", "sample": long_sample, "kind": "mean"},
            {"name": "D3 ridge lambda ladder", "sample": np.column_stack([np.ones(20), np.linspace(-1.0, 1.0, 20)]), "target": 1.0 + np.linspace(-1.0, 1.0, 20), "lambdas": np.array([0.1, 0.3, 1.0, 3.0]), "kind": "ridge"},
            {"name": "D4 compare mean/ridge to 1-NN-style rule", "sample": np.linspace(0.0, 1.0, 20), "kind": "nn_compare"},
            {"name": "D5 outlier removal stress case", "sample": np.array([0.0, 0.1, 0.2, 0.3, 9.0]), "kind": "outlier"},
        ]
    if topic == "nfl":
        return [
            {"name": "D1 one unseen point", "n": 1, "prediction": np.array([0])},
            {"name": "D2 lesson three unseen points", "n": 3, "prediction": np.array([0, 0, 0])},
            {"name": "D3 four unseen points", "n": 4, "prediction": np.array([1, 0, 1, 0])},
            {"name": "D4 biased smooth subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "smooth"},
            {"name": "D5 adversarial mirror subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "mirror"},
        ]
    if topic == "uniform":
        return [
            {"name": "D1 one fixed hypothesis", "H": 1, "m": 500, "epsilon": 0.1},
            {"name": "D2 hundred-hypothesis lesson class", "H": 100, "m": 500, "epsilon": 0.1},
            {"name": "D3 low-m vacuous rungs", "H": 100, "ms": np.array([100, 150, 230, 300]), "epsilon": 0.1},
            {"name": "D4 solve m for delta", "H": 100, "delta": 0.05, "epsilon": 0.1},
            {"name": "D5 correlated non-iid violation", "H": 100, "m": 500, "epsilon": 0.1, "rho": 0.9},
        ]
    raise ValueError(topic)


def all_binary_targets(n):
    rows = list(itertools.product([0, 1], repeat=n))
    return np.array(rows, dtype=int)


def ridge_weights(X, y, lam, penalize_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    penalty = np.eye(X.shape[1])
    if not penalize_intercept:
        penalty[0, 0] = 0.0
    system = X.T @ X + lam * penalty
    rhs = X.T @ y
    weights = np.linalg.solve(system, rhs)
    return weights


def print_rows(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, item in enumerate(row):
            widths[i] = max(widths[i], len(str(item)))
    header = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    print(header)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(item).ljust(widths[i]) for i, item in enumerate(row)))


## The concept, built once (D1)

A uniformly $\beta$-stable algorithm changes its loss by at most $\beta$ when one training point is replaced:
$$\sup_z|\ell(A(S),z)-\ell(A(S^{(i)}),z)|\le\beta.$$
For the lesson sample $\{2,4,6,8,10\}$, leave-one-out mean sensitivity should have full mean $6.0$ and max swing $1.0$.

In [ ]:

def mean_estimator(sample):
    return float(np.mean(sample))


def leave_one_out_sensitivity(estimator, sample):
    sample = np.asarray(sample, dtype=float)
    full_fit = estimator(sample)
    deleted_fits = []
    changes = []
    for i in range(len(sample)):
        reduced = np.delete(sample, i, axis=0)
        fit = estimator(reduced)
        deleted_fits.append(fit)
        changes.append(abs(float(np.asarray(fit).mean()) - float(np.asarray(full_fit).mean())))
    return full_fit, np.array(deleted_fits), np.array(changes)


Compute the exact lesson rows: dropping $2$ gives $7.0$, then $6.5,6.0,5.5,5.0$, with largest swing $1.0$.

In [ ]:

lesson_sample = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
full_mean, deleted_means, mean_changes = leave_one_out_sensitivity(mean_estimator, lesson_sample)
print(f"Full mean: {full_mean:.1f}")
print_rows([(lesson_sample[i], f"{deleted_means[i]:.1f}", f"{mean_changes[i]:.1f}") for i in range(len(lesson_sample))], ["dropped", "loo mean", "swing"])
assert np.isclose(full_mean, 6.0)
assert np.allclose(deleted_means, np.array([7.0, 6.5, 6.0, 5.5, 5.0]))
assert np.isclose(np.max(mean_changes), 1.0)


## The dataset ladder

D1 is the lesson mean. D2 shows $m=50$ bounded-range stability, D3 studies ridge as $\beta\propto1/(\lambda m)$, D4 compares stable rules with a 1-NN-style rule, and D5 stresses outlier removal.

In [ ]:

stab_ladder = theory_ladder("stability")
rows = []
for dataset in stab_ladder:
    shape = np.asarray(dataset["sample"]).shape
    rows.append((dataset["name"], dataset["kind"], shape))
print_rows(rows, ["dataset", "kind", "sample shape"])


In [ ]:

def ridge_leave_one_out_sensitivity(X, y, lambdas):
    rows = []
    for lam in lambdas:
        full = ridge_weights(X, y, lam)
        changes = []
        for i in range(len(y)):
            X_reduced = np.delete(X, i, axis=0)
            y_reduced = np.delete(y, i, axis=0)
            reduced = ridge_weights(X_reduced, y_reduced, lam)
            changes.append(float(np.linalg.norm(full - reduced)))
        rows.append((lam, max(changes), 1.0 / (lam * len(y))))
    return rows


def nearest_neighbor_flip_score(sample):
    sample = np.asarray(sample, dtype=float)
    query = 0.52
    labels = (sample > 0.5).astype(float)
    nearest = int(np.argmin(np.abs(sample - query)))
    full_prediction = labels[nearest]
    flips = []
    for i in range(len(sample)):
        reduced_sample = np.delete(sample, i)
        reduced_labels = np.delete(labels, i)
        reduced_nearest = int(np.argmin(np.abs(reduced_sample - query)))
        flips.append(abs(reduced_labels[reduced_nearest] - full_prediction))
    return max(flips)

stab_results = []
for dataset in stab_ladder:
    if dataset["kind"] in ["mean", "outlier"]:
        full, deleted, changes = leave_one_out_sensitivity(mean_estimator, dataset["sample"])
        metric = float(np.max(changes))
        extra = float(1.0 / len(dataset["sample"]))
    elif dataset["kind"] == "ridge":
        ridge_rows = ridge_leave_one_out_sensitivity(dataset["sample"], dataset["target"], dataset["lambdas"])
        metric = float(min(row[1] for row in ridge_rows))
        extra = float(min(row[2] for row in ridge_rows))
    else:
        metric = float(nearest_neighbor_flip_score(dataset["sample"]))
        extra = 0.0
    stab_results.append({"dataset": dataset, "metric": metric, "proxy": extra})
    print(f"{dataset['name']}: sensitivity {metric:.3f}, schematic proxy {extra:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for j, result in enumerate(stab_results):
    dataset = result["dataset"]
    if dataset["kind"] in ["mean", "outlier"]:
        full, deleted, changes = leave_one_out_sensitivity(mean_estimator, dataset["sample"])
        axes[0, j].plot(np.arange(len(deleted)), deleted, marker="o")
        axes[0, j].axhline(full, color="black", linestyle="--")
        axes[0, j].set_ylabel("fit after deletion")
    elif dataset["kind"] == "ridge":
        ridge_rows = ridge_leave_one_out_sensitivity(dataset["sample"], dataset["target"], dataset["lambdas"])
        axes[0, j].plot([row[0] for row in ridge_rows], [row[1] for row in ridge_rows], marker="o")
        axes[0, j].set_xscale("log")
        axes[0, j].set_ylabel("weight swing")
    else:
        labels = (dataset["sample"] > 0.5).astype(float)
        axes[0, j].scatter(dataset["sample"], labels)
        axes[0, j].axvline(0.52, color="black", linestyle="--")
    axes[0, j].set_title(dataset["name"].split()[0])
metrics = np.array([result["metric"] for result in stab_results])
proxies = np.array([result["proxy"] for result in stab_results])
axes[1, 0].plot(np.arange(1, 6), metrics, marker="o", label="empirical sensitivity")
axes[1, 0].plot(np.arange(1, 6), proxies, marker="o", label="1/m or 1/(lambda m)")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("generalization-gap proxy")
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis("off")
fig.suptitle("Stability capacity panels and sensitivity-vs-sample/regularization curve")
plt.tight_layout()


## Pitfall on D5: assuming every algorithm is stable

1-nearest-neighbor can change its prediction near a boundary when one point is deleted. The fix is not to claim a stability guarantee; either add regularization/use a stable learner or report that this route provides no bound.

In [ ]:

nn_flip = nearest_neighbor_flip_score(stab_ladder[3]["sample"])
outlier_full, outlier_deleted, outlier_changes = leave_one_out_sensitivity(mean_estimator, stab_ladder[-1]["sample"])
trimmed_sample = np.clip(stab_ladder[-1]["sample"], 0.0, 1.0)
trimmed_full, trimmed_deleted, trimmed_changes = leave_one_out_sensitivity(mean_estimator, trimmed_sample)
print(f"1-NN-style deletion flip score: {nn_flip:.1f}")
print(f"Outlier mean max swing: {np.max(outlier_changes):.3f}")
print(f"Winsorized mean max swing: {np.max(trimmed_changes):.3f}")


## Evaluate it + practice

- Metric: leave-one-out sensitivity or a stability-shaped generalization-gap proxy.
- No-skill baseline: a constant predictor has zero sensitivity but may be useless.
- Cheap sanity check: mean sensitivity should shrink as sample size grows.
- Ablation: remove regularization in ridge and compare larger deletion swings.
- Failure signals: a nearest-neighbor rule flips or a stable but useless learner wins the gap metric.

### Practice prompts

- Repeat the mean sensitivity calculation after adding one bounded point.

- Double every ridge lambda and check the schematic proxy.

- Move the 1-NN query and find where deletion flips disappear.
